In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression ,ElasticNet
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import OrdinalEncoder # Importing OrdinalEncoder from sklearn.preprocessing
from sklearn.svm import SVC
import scipy.stats as stats
from sklearn.metrics import accuracy_score

In [ ]:
test = pd.read_csv('/content/test.csv')
train = pd.read_csv('/content/train.csv')
submission = pd.read_csv('/content/sample_submission.csv')
# training_extra.csv was not used by the original estimator.

In [ ]:
train.head()

In [ ]:
train.info()

In [ ]:
train.isna().sum()

In [ ]:
train = train.dropna(subset=['Price']).copy()
train.shape

In [ ]:
test.isna().sum()

In [ ]:
submission.head()

In [ ]:
assert train['id'].is_unique and test['id'].is_unique
train.shape, test.shape

In [ ]:
for col in train:
    if train[col].dtype == 'object':
        print(col,train[col].unique() )

In [ ]:
plt.hist(train['Style'].dropna().astype(str))
plt.xticks(rotation=90)
plt.show()

In [ ]:

color_counts = train['Color'].value_counts()

# إنشاء مخطط الأعمدة
plt.figure(figsize=(10, 5))  # ضبط حجم الشكل لتحسين الوضوح
plt.bar(color_counts.index, color_counts.values, color='skyblue')  # رسم المخطط مع لون مميز


plt.xticks(rotation=90)
plt.xlabel("Color")
plt.ylabel("Count")
plt.title("Distribution of Colors in Train Data")

# عرض المخطط
plt.show()

In [ ]:
sns.displot(train['Price'], kde=True)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
features = [c for c in train.columns if c not in ['id', 'Price']]
if set(features) != set(test.columns) - {'id'}:
    raise ValueError('Training and test features differ')

In [ ]:
train.info()

In [ ]:
train


In [ ]:
test.info()

In [ ]:
y = train['Price']
X = train[features].copy()
X_test = test[features].copy()

In [ ]:
X_train, X_val, y_train, y_val = train_test_split( X, y, test_size=0.1, shuffle=True, random_state=42)
X_train.shape, y_train.shape, X_val.shape, y_val.shape, X_test.shape

In [ ]:
cat = X_train.select_dtypes(include=['object', 'category', 'string', 'bool']).columns.tolist()
num = [c for c in features if c not in cat]
preprocess = ColumnTransformer([
    ('numeric', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), num),
    ('categorical', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('encode', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))]), cat),
])
model = Pipeline([('preprocess', preprocess), ('regressor', ElasticNet(random_state=42))])
model.fit(X_train, y_train)
model.score(X_train, y_train)

In [ ]:
y_pred = model.predict(X_val)
y_pred

In [ ]:
mse = mean_squared_error(y_val, y_pred)
rmse = np.sqrt(mse)
rmse

In [ ]:
df = pd.DataFrame({'y_val':y_val, 'y_pred':y_pred})
df

In [ ]:
stats.probplot(y_pred, dist="norm", plot=plt)
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.scatter(y_val, y_pred, edgecolors=(0, 0, 0))
ax.plot([y.min(), y.max()], [y.min(), y.max()], 'k--', lw=4)
ax.set_xlabel('Measured')
ax.set_ylabel('Predicted')
plt.show()

In [ ]:
from sklearn.base import clone
# Refit only after held-out validation, for final inference on the external test set.
final_model = clone(model).fit(X, y)
pred = final_model.predict(X_test)
pred

In [ ]:
if not submission['id'].is_unique or set(submission['id']) != set(test['id']):
    raise ValueError('Sample submission IDs do not match test IDs')
if len(pred) != len(test) or not np.isfinite(pred).all():
    raise ValueError('Predictions must be finite and match the test rows')
by_id = pd.Series(pred, index=test['id'])
submission['Price'] = submission['id'].map(by_id)
assert submission['Price'].notna().all()
submission.to_csv('submission.csv', index=False)
print('Final Submission Shape:', submission.shape)